# Figure 1c: AEM resistivity

Draws the regional airborne-electromagnetic resistivity section.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial import cKDTree


MIN_ACTIVE_CELLS_PER_ROW = 12
CENTERLINE_SMOOTH_ROWS = 21
TOPO_SMOOTH_ROWS = 11
DEPTH_MAX_M = 350.0

FIG_WIDTH_IN = 7.2
FIG_HEIGHT_IN = 2
DPI = 600
FRAME_LW = 0.7
TICK_LW = 0.65
CBAR_VMIN = -0.41
CBAR_VMAX = 3.29

AX_POS = [0.085, 0.17, 0.765, 0.70]
CBAR_POS = [0.885, 0.17, 0.022, 0.70]


ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = next(p for p in ROOT.parents if (p / "data").exists())

AEM_NC = ROOT / "data_raw" / "1 MAP Electrical Resistivity & Facies Classification" / "MAP_RegionalAEM_2022_ResistivityFacies_DepthGrids_log10res_masked_latestmask.nc"
ACTIVE_CSV = ROOT / "data" / "2 well WTD" / "active_cell_lookup.csv"
TOPO_CSV = ROOT / "data" / "5 topography" / "topo_1km.csv"
OUT_PNG = ROOT / "outputs" / "figures" / "Fig1c_resistivity" / "Fig1c_resistivity.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)


active = pd.read_csv(ACTIVE_CSV)
topo = pd.read_csv(TOPO_CSV)
active_topo = active.merge(topo, left_on="node_id", right_on="grid_id", how="left", validate="one_to_one")

centerline = (
    active_topo.groupby("row", as_index=False)
    .agg(y=("y_center", "median"), x_median=("x_center", "median"), n_active=("node_id", "size"))
    .query("n_active >= @MIN_ACTIVE_CELLS_PER_ROW")
    .sort_values("y")
    .reset_index(drop=True)
)
centerline["x"] = centerline["x_median"].rolling(CENTERLINE_SMOOTH_ROWS, center=True, min_periods=1).median()

tree = cKDTree(active_topo[["x_center", "y_center"]].to_numpy(float))
_, topo_idx = tree.query(centerline[["x", "y"]].to_numpy(float), k=1)
centerline["surface_elevation_m"] = (
    pd.Series(active_topo["mean_elevation_m"].to_numpy(float)[topo_idx])
    .rolling(TOPO_SMOOTH_ROWS, center=True, min_periods=1)
    .mean()
)

xy = centerline[["x", "y"]].to_numpy(float)
centerline["distance_km"] = np.r_[0.0, np.cumsum(np.hypot(np.diff(xy[:, 0]), np.diff(xy[:, 1])))] / 1000.0

with xr.open_dataset(AEM_NC, engine="netcdf4") as ds:
    depths = ds["z"].to_numpy().astype(float)
    depth_mask = depths <= DEPTH_MAX_M + 1e-6
    xq = xr.DataArray(centerline["x"].to_numpy(float), dims="point")
    yq = xr.DataArray(centerline["y"].to_numpy(float), dims="point")
    section = ds["log10res"].sel(x=xq, y=yq, method="nearest").transpose("z", "point").to_numpy().astype(float)
    section = np.where(section > 1000.0, np.nan, section)[depth_mask]
    z_bnds = ds["z_bnds"].to_numpy()[depth_mask]
    depth_edges = np.r_[z_bnds[0, 0], z_bnds[:, 1]]


mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
        "font.size": 7.0,
        "axes.linewidth": FRAME_LW,
        "axes.edgecolor": "black",
        "xtick.major.width": TICK_LW,
        "ytick.major.width": TICK_LW,
        "axes.spines.top": True,
        "axes.spines.right": True,
    }
)

def cell_edges(values):
    values = np.asarray(values, dtype=float)
    return np.r_[values[0] - 0.5 * (values[1] - values[0]), 0.5 * (values[:-1] + values[1:]), values[-1] + 0.5 * (values[-1] - values[-2])]

colors = [
    (CBAR_VMIN, "#1b1bb3"),
    (0.00, "#2445b8"),
    (0.50, "#55b8e8"),
    (1.00, "#d7f4ed"),
    (1.50, "#fff7bc"),
    (2.00, "#f6c066"),
    (2.50, "#e66b2e"),
    (3.00, "#c93422"),
    (CBAR_VMAX, "#b2181b"),
]
cmap = LinearSegmentedColormap.from_list(
    "fig1c_reference_log10res",
    [((v - CBAR_VMIN) / (CBAR_VMAX - CBAR_VMIN), c) for v, c in colors],
    N=256,
)

distance = centerline["distance_km"].to_numpy(float)
surface = centerline["surface_elevation_m"].to_numpy(float)
distance_edges = cell_edges(distance)
surface_edges = cell_edges(surface)
x_edges = np.tile(distance_edges, (len(depth_edges), 1))
z_edges = surface_edges[None, :] - depth_edges[:, None]
y_top = float(np.ceil((np.nanmax(surface) + 28.0) / 25.0) * 25.0)
y_bottom = float(np.floor((np.nanmin(surface) - DEPTH_MAX_M - 20.0) / 25.0) * 25.0)

fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN), dpi=600)
ax = fig.add_axes(AX_POS)
cax = fig.add_axes(CBAR_POS)

mesh = ax.pcolormesh(x_edges, z_edges, np.ma.masked_invalid(section), cmap=cmap, vmin=CBAR_VMIN, vmax=CBAR_VMAX, shading="auto", rasterized=True)
ax.fill_between(distance, surface, y_top, color="white", zorder=3)
ax.plot(distance, surface, color="white", lw=2.0, solid_capstyle="round", zorder=4)
ax.plot(distance, surface, color="#222222", lw=0.95, solid_capstyle="round", zorder=5)
ax.set(xlim=(distance_edges[0], distance_edges[-1]), ylim=(y_bottom, y_top), xlabel="", ylabel="")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.tick_params(axis="both", which="both", length=0, labelbottom=False, labelleft=False)

cbar = fig.colorbar(mesh, cax=cax, extend="both", extendfrac=0.075)
cbar.set_ticks([])
cbar.outline.set_linewidth(FRAME_LW)
cbar.outline.set_edgecolor("black")

fig.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
print(f"Saved PNG: {OUT_PNG}")